# Enhanced py-ard: Donor-Recipient Compatibility Analysis

This notebook demonstrates donor-recipient compatibility analysis using py-ard with the [HLAtools](https://github.com/sjmack/HLAtools) R package for authoritative alignment data.

## Features Demonstrated
- HLAtools R package integration via `pyard.alignment_bridge` for authoritative alignment data
- Real amino acid sequence analysis with per-residue IMGT position numbering
- Accurate leader/mature protein boundary classification using IMGT positions directly (negative = leader, positive = mature)
- Donor-recipient genotype compatibility assessment
- Detailed mismatch analysis with biological context
- CSV export with IMGT position and sequence region annotations
- Multi-locus support (A, B, C, DRB1, DPB1, DQB1, and more)
- **Mature protein redux (M redux)**: ambiguous alleles that differ only in the leader peptide are collapsed to a single representative annotated with `[M]`
- **Identical AAMM check**: determines whether all possible phasing combinations yield the same amino acid mismatch set, so the clinical assessment is unambiguous regardless of true allele assignment
- **BDD validation**: each feature is backed by a formal Gherkin test suite in `tests/features/` (run inline in [Test 9](#test-9-bdd-feature-tests))

## Prerequisites
- **R** (>= 3.6) with the **HLAtools** package installed: `Rscript -e "install.packages('HLAtools')"`
- **rpy2** and **pandas** Python packages: `pip install rpy2 pandas`
- py-ard project root on `sys.path` (handled automatically by the first code cell)

## Leader/Mature Boundary Resolution

HLAtools provides IMGT position numbers as data frame column headers, giving exact position mapping with no interpolation needed. Negative positions are leader peptide, positive positions are mature protein. For DPB1, the boundary falls cleanly at IMGT position -1 (leader) to position 1 (mature), with 29 leader and 244 mature positions.

## Setup

### Dependency Check

Run the cell below **first** to verify that all required packages are installed and accessible. The cell checks:

| Dependency | How to install if missing |
|---|---|
| Python ≥ 3.9 | — |
| **rpy2** | `pip install rpy2` |
| **pandas** | `pip install pandas` |
| **R** (≥ 3.6) | [https://cran.r-project.org](https://cran.r-project.org) |
| **HLAtools** R package | `Rscript -e "install.packages('HLAtools')"` |
| **py-ard** (this repo) | ensure the repo root is on `sys.path` (handled automatically) |

In [1]:
import importlib
import subprocess
import sys
import os

print("=" * 60)
print("Dependency Check")
print("=" * 60)

ok = True

# Python version
major, minor = sys.version_info[:2]
ver_str = f"Python {major}.{minor}"
if (major, minor) >= (3, 9):
    print(f"[OK]     {ver_str}")
else:
    print(f"[WARN]   {ver_str}  (>= 3.9 recommended)")

# Python packages
for pkg in ["rpy2", "pandas"]:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "installed")
        print(f"[OK]     {pkg} {version}")
    except ImportError:
        print(f"[MISSING] {pkg}  ->  pip install {pkg}")
        ok = False

# R installation
try:
    r_proc = subprocess.run(
        ["Rscript", "--version"], capture_output=True, text=True, timeout=15
    )
    r_ver = (r_proc.stdout or r_proc.stderr).strip().splitlines()[0]
    print(f"[OK]     R  ({r_ver})")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("[MISSING] R  ->  https://cran.r-project.org")
    ok = False

# HLAtools R package (only check if rpy2 + R are available)
try:
    import rpy2.robjects as ro
    ro.r("library(HLAtools)")
    print("[OK]     HLAtools R package")
except Exception:
    print('[MISSING] HLAtools  ->  Rscript -e "install.packages(\'HLAtools\')"')
    ok = False

# py-ard on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
try:
    import pyard
    print(f"[OK]     pyard {pyard.__version__}")
except ImportError:
    print("[MISSING] pyard  ->  add py-ard project root to sys.path")
    ok = False

print()
if ok:
    print("All dependencies satisfied — ready to run the notebook.")
else:
    print("One or more dependencies are missing. Please install them before continuing.")

Dependency Check
[OK]     Python 3.13
[OK]     rpy2 installed


[OK]     pandas 3.0.0
[OK]     R  (Rscript (R) version 4.5.1 (2025-06-13))


[OK]     HLAtools R package
[OK]     pyard 2.0.0b2

All dependencies satisfied — ready to run the notebook.


In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from pyard.alignment_bridge import HLAToolsBridge

bridge = HLAToolsBridge()
if not bridge.is_available:
    raise RuntimeError(
        "HLAtools R package is required.\n"
        'Install with: pip install rpy2 && Rscript -e "install.packages(\'HLAtools\')"'
    )

print("HLAtools bridge ready.")
print("Note: First call to get_alignment() will load data (may take several minutes)")

HLAtools bridge ready.
Note: First call to get_alignment() will load data (may take several minutes)


## Test 1: Load and Examine Protein Alignment Data

In [3]:
print("Testing Enhanced Protein Alignment Loading")
print("=" * 50)

dpb1_alignment = bridge.get_alignment("DPB1")

if dpb1_alignment:
    print(f"\nSuccessfully loaded {len(dpb1_alignment)} DPB1 protein sequences")

    print("\nExample sequences:")
    for allele, sequence in list(dpb1_alignment.items())[:5]:
        print(f"  {allele}: {sequence[:60]}... (length: {len(sequence)})")

    test_alleles = ["DPB1*04:01:01:01", "DPB1*04:02:01:01", "DPB1*05:01:01:01"]
    print(f"\nChecking for specific test alleles:")
    for allele in test_alleles:
        if allele in dpb1_alignment:
            print(f"  {allele}: Available (length {len(dpb1_alignment[allele])})")
        else:
            similar = [a for a in dpb1_alignment.keys() if a.startswith(allele.split(":")[0])]
            if similar:
                print(f"  {allele}: Not found, using {similar[0]} instead")
            else:
                print(f"  {allele}: Not available")
else:
    print("Failed to load DPB1 alignment data")

Testing Enhanced Protein Alignment Loading



Successfully loaded 2889 DPB1 protein sequences

Example sequences:
  DPB1*01:01:01:01: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:02: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:03: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:04: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:05: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)

Checking for specific test alleles:
  DPB1*04:01:01:01: Available (length 258)
  DPB1*04:02:01:01: Available (length 258)
  DPB1*05:01:01:01: Available (length 258)


## Core Analysis Functions

In [4]:
def get_amino_acid_sequence(allele: str):
    """Get amino acid sequence for an allele via HLAtools bridge."""
    seq = bridge.get_sequence(allele)
    return (seq, "hlatools") if seq is not None else (None, "error")


def compare_single_alleles(donor_allele: str, recipient_allele: str) -> dict:
    """Compare two alleles for amino acid mismatches."""
    donor_seq, donor_method = get_amino_acid_sequence(donor_allele)
    recipient_seq, recipient_method = get_amino_acid_sequence(recipient_allele)

    if not donor_seq or not recipient_seq:
        return {
            "error": f"Could not retrieve sequences for {donor_allele} and/or {recipient_allele}",
            "donor_allele": donor_allele,
            "recipient_allele": recipient_allele,
            "mismatch_count": None,
        }

    mismatches = []
    for i, (donor_aa, recipient_aa) in enumerate(zip(donor_seq, recipient_seq)):
        if donor_aa != recipient_aa:
            mismatches.append({
                "position": i + 1,
                "donor_aa": donor_aa,
                "recipient_aa": recipient_aa,
                "change": f"{donor_aa}>{recipient_aa}",
            })

    return {
        "donor_allele": donor_allele,
        "recipient_allele": recipient_allele,
        "mismatch_count": len(mismatches),
        "mismatch_positions": [m["position"] for m in mismatches],
        "substitutions": [m["change"] for m in mismatches],
        "mismatches": mismatches,
        "donor_sequence_length": len(donor_seq),
        "recipient_sequence_length": len(recipient_seq),
        "analysis_method": f"{donor_method}/{recipient_method}",
    }


print("Core analysis functions loaded")

Core analysis functions loaded


## Test 2: Single Allele Comparison

In [5]:
print("Testing Single Allele Comparison")
print("=" * 40)

donor_allele = "DPB1*04:01:01:01"
recipient_allele = "DPB1*04:02:01:01"

print(f"Comparing {donor_allele} vs {recipient_allele}")
result = compare_single_alleles(donor_allele, recipient_allele)

if not result.get("error"):
    print(f"\nComparison Results:")
    print(f"  Donor: {result['donor_allele']} (length: {result['donor_sequence_length']})")
    print(f"  Recipient: {result['recipient_allele']} (length: {result['recipient_sequence_length']})")
    print(f"  Mismatches: {result['mismatch_count']}")
    print(f"  Analysis method: {result['analysis_method']}")

    if result['mismatch_count'] > 0:
        print(f"  Mismatch positions: {result['mismatch_positions'][:10]}{'...' if len(result['mismatch_positions']) > 10 else ''}")
        print(f"  Substitutions: {result['substitutions'][:10]}{'...' if len(result['substitutions']) > 10 else ''}")
        print(f"\nFirst 5 detailed mismatches:")
        for mismatch in result['mismatches'][:5]:
            print(f"    Position {mismatch['position']}: {mismatch['change']}")
    else:
        print(f"  No mismatches found - sequences are identical!")
else:
    print(f"Error: {result['error']}")

Testing Single Allele Comparison
Comparing DPB1*04:01:01:01 vs DPB1*04:02:01:01

Comparison Results:
  Donor: DPB1*04:01:01:01 (length: 258)
  Recipient: DPB1*04:02:01:01 (length: 258)
  Mismatches: 4
  Analysis method: hlatools/hlatools
  Mismatch positions: [65, 84, 85, 207]
  Substitutions: ['A>V', 'A>D', 'A>E', 'L>M']

First 5 detailed mismatches:
    Position 65: A>V
    Position 84: A>D
    Position 85: A>E
    Position 207: L>M


## Test 3: Position Mapping and Leader/Mature Classification

In [6]:
print("Testing Position Mapping for Leader/Mature Classification")
print("=" * 60)

locus = "DPB1"
position_mapping = bridge.get_position_mapping(locus)

if position_mapping:
    print(f"\nSuccessfully created position mapping with {len(position_mapping)} positions")

    print("\nExample sequence index -> IMGT position mappings:")
    for seq_idx, imgt_pos in list(position_mapping.items())[:20]:
        region = "Leader" if imgt_pos < 1 else "Mature"
        print(f"  Sequence position {seq_idx+1:3d} -> IMGT position {imgt_pos:3d} ({region})")

    leader_positions = [pos for pos in position_mapping.values() if pos < 1]
    mature_positions = [pos for pos in position_mapping.values() if pos >= 1]

    print(f"\nSequence region summary:")
    print(f"  Leader sequence positions: {len(leader_positions)}")
    print(f"  Mature protein positions: {len(mature_positions)}")
    print(f"  Total mapped positions: {len(position_mapping)}")

    boundary_positions = [(idx, pos) for idx, pos in position_mapping.items() if -5 <= pos <= 5]
    if boundary_positions:
        print(f"\nLeader/Mature boundary region:")
        for seq_idx, imgt_pos in sorted(boundary_positions):
            region = "Leader" if imgt_pos < 1 else "Mature"
            print(f"  Sequence position {seq_idx+1:3d} -> IMGT position {imgt_pos:3d} ({region})")
else:
    print("Failed to create position mapping")

Testing Position Mapping for Leader/Mature Classification



Successfully created position mapping with 273 positions

Example sequence index -> IMGT position mappings:
  Sequence position   1 -> IMGT position -29 (Leader)
  Sequence position   2 -> IMGT position -28 (Leader)
  Sequence position   3 -> IMGT position -27 (Leader)
  Sequence position   4 -> IMGT position -26 (Leader)
  Sequence position   5 -> IMGT position -25 (Leader)
  Sequence position   6 -> IMGT position -24 (Leader)
  Sequence position   7 -> IMGT position -23 (Leader)
  Sequence position   8 -> IMGT position -22 (Leader)
  Sequence position   9 -> IMGT position -21 (Leader)
  Sequence position  10 -> IMGT position -20 (Leader)
  Sequence position  11 -> IMGT position -19 (Leader)
  Sequence position  12 -> IMGT position -18 (Leader)
  Sequence position  13 -> IMGT position -17 (Leader)
  Sequence position  14 -> IMGT position -16 (Leader)
  Sequence position  15 -> IMGT position -15 (Leader)
  Sequence position  16 -> IMGT position -14 (Leader)
  Sequence position  17 -> 

## Analysis Functions

In [7]:
def parse_gl_genotype(genotype: str, locus: str) -> list:
    """
    Parse a GL string genotype into a list of haplotype groups.

    '+' separates the two chromosomes of a diplotype.
    '/' separates ambiguous allele alternatives for one chromosome.

    Examples:
        "DPB1*04:01+DPB1*04:02"
            -> [["DPB1*04:01"], ["DPB1*04:02"]]

        "DPB1*04:01/DPB1*04:02+DPB1*02:01"
            -> [["DPB1*04:01", "DPB1*04:02"], ["DPB1*02:01"]]
    """
    result = []
    for hap_str in genotype.replace("HLA-", "").strip().split("+"):
        alleles = []
        for raw in hap_str.split("/"):
            raw = raw.strip()
            if not raw:
                continue
            if "*" not in raw:
                raw = f"{locus}*{raw}"
            alleles.append(raw)
        if alleles:
            result.append(alleles)
    return result


def is_null_allele(allele: str) -> bool:
    """Return True if the allele is a null allele (expression suffix 'N')."""
    return allele.upper().endswith("N")


def analyze_transplant_mismatches(
    donor_genotype: str, recipient_genotype: str, locus: str
) -> dict:
    """
    Determine donor allele mismatches for solid organ transplant.

    A mismatch is a donor chromosome whose expressed protein sequence is
    not present in any recipient allele. Null alleles (N suffix) in the
    donor are not counted as mismatches.

    For ambiguous donor typings ('/' separated alternatives):
      'mismatch'       -- all expressed options are absent from recipient sequences
      'possible_match' -- some options match recipient, some do not (ambiguous result)
      'match'          -- all expressed options are present in recipient sequences
    """
    donor_haplotypes = parse_gl_genotype(donor_genotype, locus)
    recipient_haplotypes = parse_gl_genotype(recipient_genotype, locus)

    # Collect all distinct protein sequences present in the recipient
    recipient_seqs = set()
    for hap in recipient_haplotypes:
        for allele in hap:
            if not is_null_allele(allele):
                seq = bridge.get_sequence(allele)
                if seq is not None:
                    recipient_seqs.add(seq)

    haplotype_results = []
    for donor_hap in donor_haplotypes:
        expressed = [a for a in donor_hap if not is_null_allele(a)]

        if not expressed:
            haplotype_results.append({
                "alleles": donor_hap,
                "status": "null",
                "is_mismatch": False,
            })
            continue

        seq_map = {a: bridge.get_sequence(a) for a in expressed}
        matched = [a for a, s in seq_map.items() if s is not None and s in recipient_seqs]
        unmatched = [a for a, s in seq_map.items() if s is None or s not in recipient_seqs]

        if matched and unmatched:
            status = "possible_match"
        elif matched:
            status = "match"
        else:
            status = "mismatch"

        haplotype_results.append({
            "alleles": donor_hap,
            "status": status,
            "is_mismatch": status == "mismatch",
            "matched_alleles": matched,
            "unmatched_alleles": unmatched,
        })

    return {
        "donor_genotype": donor_genotype,
        "recipient_genotype": recipient_genotype,
        "locus": locus,
        "donor_haplotypes": donor_haplotypes,
        "recipient_haplotypes": recipient_haplotypes,
        "haplotype_results": haplotype_results,
        "mismatch_count": sum(r["is_mismatch"] for r in haplotype_results),
        "possible_mismatch_count": sum(
            1 for r in haplotype_results if r["status"] == "possible_match"
        ),
    }


print("Analysis functions loaded")

Analysis functions loaded


## Test 4: Donor-Recipient Genotype Analysis

In [8]:
print("Testing Transplant Mismatch Analysis")
print("=" * 55)

cases = [
    {
        "label": "Case 1: One definite mismatch",
        "donor":     "DPB1*04:01+DPB1*02:01",
        "recipient": "DPB1*04:01+DPB1*03:01",
        "note": "04:01 matches; 02:01 not in recipient -> 1 mismatch",
    },
    {
        "label": "Case 2: Perfect match",
        "donor":     "DPB1*04:01+DPB1*03:01",
        "recipient": "DPB1*04:01+DPB1*03:01",
        "note": "Both donor alleles present in recipient -> 0 mismatches",
    },
    {
        "label": "Case 3: Ambiguous donor chr1 (04:01/04:02), one definite mismatch on chr2",
        "donor":     "DPB1*04:01/04:02+DPB1*02:01",
        "recipient": "DPB1*04:01+DPB1*03:01",
        "note": "chr1: 04:01 matches -> possible_match; chr2: 02:01 absent -> mismatch",
    },
    {
        "label": "Case 4: Null allele on donor chr2 (not a mismatch)",
        "donor":     "DPB1*04:01+DPB1*02:01N",
        "recipient": "DPB1*04:01+DPB1*03:01",
        "note": "04:01 matches; 02:01N is null -> skipped, not a mismatch",
    },
]

mismatch_result = None

for case in cases:
    print(f"\n{case['label']}")
    print(f"  ({case['note']})")
    print(f"  Donor:     {case['donor']}")
    print(f"  Recipient: {case['recipient']}")
    result = analyze_transplant_mismatches(case["donor"], case["recipient"], "DPB1")
    print(f"  Definite mismatches: {result['mismatch_count']}"
          + (f"  |  Possible mismatches: {result['possible_mismatch_count']}"
             if result["possible_mismatch_count"] else ""))
    for r in result["haplotype_results"]:
        allele_str = "/".join(r["alleles"])
        line = f"    [{allele_str}] -> {r['status']}"
        if r.get("matched_alleles"):
            line += f"  (matched: {r['matched_alleles']})"
        if r.get("unmatched_alleles") and r["status"] != "match":
            line += f"  (unmatched: {r['unmatched_alleles']})"
        print(line)

# Keep Case 3 result for the CSV export test below
mismatch_result = analyze_transplant_mismatches(
    "DPB1*04:01/04:02+DPB1*02:01",
    "DPB1*04:01+DPB1*03:01",
    "DPB1",
)

Testing Transplant Mismatch Analysis

Case 1: One definite mismatch
  (04:01 matches; 02:01 not in recipient -> 1 mismatch)
  Donor:     DPB1*04:01+DPB1*02:01
  Recipient: DPB1*04:01+DPB1*03:01
  Definite mismatches: 1
    [DPB1*04:01] -> match  (matched: ['DPB1*04:01'])
    [DPB1*02:01] -> mismatch  (unmatched: ['DPB1*02:01'])

Case 2: Perfect match
  (Both donor alleles present in recipient -> 0 mismatches)
  Donor:     DPB1*04:01+DPB1*03:01
  Recipient: DPB1*04:01+DPB1*03:01
  Definite mismatches: 0
    [DPB1*04:01] -> match  (matched: ['DPB1*04:01'])
    [DPB1*03:01] -> match  (matched: ['DPB1*03:01'])

Case 3: Ambiguous donor chr1 (04:01/04:02), one definite mismatch on chr2
  (chr1: 04:01 matches -> possible_match; chr2: 02:01 absent -> mismatch)
  Donor:     DPB1*04:01/04:02+DPB1*02:01
  Recipient: DPB1*04:01+DPB1*03:01
  Definite mismatches: 1  |  Possible mismatches: 1
    [DPB1*04:01/DPB1*04:02] -> possible_match  (matched: ['DPB1*04:01'])  (unmatched: ['DPB1*04:02'])
    [DP

## Test 5: CSV Export with Position Classification

In [9]:
def export_mismatch_analysis_to_csv(
    mismatch_result: dict, position_mapping: dict, filename: str
) -> str:
    """
    Export transplant mismatch analysis to CSV.

    For mismatched/possible_match donor haplotypes, the amino acid differences
    against each recipient allele are written as individual rows.
    """
    import csv

    recipient_alleles = [
        a
        for hap in mismatch_result["recipient_haplotypes"]
        for a in hap
        if not is_null_allele(a)
    ]

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Donor_Haplotype",
            "Mismatch_Status",
            "Recipient_Allele",
            "AA_Differences",
            "Position",
            "Substitution",
            "Sequence_Region",
            "IMGT_Position",
        ])

        for hap_result in mismatch_result["haplotype_results"]:
            donor_str = "/".join(hap_result["alleles"])
            status = hap_result["status"]

            if status == "null":
                writer.writerow([donor_str, "null", "N/A", 0, "N/A", "N/A", "N/A", "N/A"])
                continue

            if status == "match":
                writer.writerow([donor_str, "match", "", 0, "None", "None", "None", "None"])
                continue

            # For mismatches/possible_match: compare each expressed donor allele
            # against each recipient allele to detail the amino acid differences
            expressed = [a for a in hap_result["alleles"] if not is_null_allele(a)]
            for donor_allele in expressed:
                for recip_allele in recipient_alleles:
                    comp = compare_single_alleles(donor_allele, recip_allele)
                    if comp.get("error"):
                        continue
                    if comp["mismatch_count"] == 0:
                        writer.writerow([donor_str, status, recip_allele, 0, "None", "None", "None", "None"])
                    else:
                        for pos, sub in zip(comp["mismatch_positions"], comp["substitutions"]):
                            imgt_pos = position_mapping.get(pos - 1)
                            if imgt_pos is not None:
                                region = "Leader" if imgt_pos < 1 else "Mature"
                                imgt_str = str(imgt_pos)
                            else:
                                region = "Unknown"
                                imgt_str = "Unknown"
                            writer.writerow([
                                donor_str, status, recip_allele,
                                comp["mismatch_count"], pos, sub, region, imgt_str,
                            ])

    return f"Results exported to {filename}"


print("Testing CSV Export with Mismatch Analysis")
print("=" * 50)

csv_filename = "standalone_demo_results.csv"
export_status = export_mismatch_analysis_to_csv(mismatch_result, position_mapping, csv_filename)
print(export_status)

with open(csv_filename, "r") as f:
    lines = f.readlines()[:20]
print(f"\nFirst 20 lines of {csv_filename}:")
for i, line in enumerate(lines, 1):
    print(f"{i:2d}: {line.strip()}")

Testing CSV Export with Mismatch Analysis
Results exported to standalone_demo_results.csv

First 20 lines of standalone_demo_results.csv:
 1: Donor_Haplotype,Mismatch_Status,Recipient_Allele,AA_Differences,Position,Substitution,Sequence_Region,IMGT_Position
 2: DPB1*04:01/DPB1*04:02,possible_match,DPB1*04:01,0,None,None,None,None
 3: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,37,L>V,Mature,8
 4: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,38,F>Y,Mature,9
 5: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,40,G>L,Mature,11
 6: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,65,A>V,Mature,29
 7: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,84,A>D,Mature,48
 8: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,85,A>E,Mature,49
 9: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,86,E>D,Mature,50
10: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,94,I>L,Mature,58
11: DPB1*04:01/DPB1*04:02,possible_match,DPB1*03:01,15,105,M>V,Mature,69
12: DPB1*04:01/DPB1*0

## Test 6: Multiple Loci Demonstration

In [10]:
print("Testing Multiple HLA Loci")
print("=" * 35)

test_loci = ["A", "B", "C", "DRB1"]

for locus in test_loci:
    print(f"\nTesting {locus} locus:")
    try:
        alignment = bridge.get_alignment(locus)
        if alignment and len(alignment) >= 2:
            alleles = list(alignment.keys())[:2]
            print(f"  Loaded {len(alignment)} alleles")
            print(f"  Sample comparison: {alleles[0]} vs {alleles[1]}")
            comp_result = compare_single_alleles(alleles[0], alleles[1])
            if not comp_result.get('error'):
                print(f"  Result: {comp_result['mismatch_count']} mismatches")
                print(f"  Sequence lengths: {comp_result['donor_sequence_length']} vs {comp_result['recipient_sequence_length']}")
            else:
                print(f"  Comparison failed: {comp_result['error']}")
        else:
            print(f"  Could not load sufficient data for {locus}")
    except Exception as e:
        print(f"  Error testing {locus}: {e}")

Testing Multiple HLA Loci

Testing A locus:


  Loaded 8715 alleles
  Sample comparison: A*01:01:01:01 vs A*01:01:01:02N
  Result: 82 mismatches
  Sequence lengths: 365 vs 200

Testing B locus:


  Loaded 10578 alleles
  Sample comparison: B*07:02:01:01 vs B*07:02:01:02
  Result: 0 mismatches
  Sequence lengths: 362 vs 362

Testing C locus:


  Loaded 8807 alleles
  Sample comparison: C*01:02:01:01 vs C*01:02:01:02
  Result: 0 mismatches
  Sequence lengths: 366 vs 366

Testing DRB1 locus:


  Loaded 3856 alleles
  Sample comparison: DRB1*01:01:01:01 vs DRB1*01:01:01:02
  Result: 0 mismatches
  Sequence lengths: 266 vs 266


## Test 7: Mature Protein Redux (M Redux)

Alleles that differ only in the leader peptide (exon 1, IMGT positions < 1) are functionally equivalent at the surface — they present the same mature protein to the immune system. The **M redux** collapses a group of ambiguous alleles with identical mature protein sequences down to the first representative, annotated with `[M]`.

> **BDD coverage**: the full set of M redux scenarios (pair collapse, partial collapse, missing reference sequence) is run as a formal Gherkin suite in [Test 9](#test-9-bdd-feature-tests) below.

In [11]:
def mature_protein_redux(alleles: list, locus: str) -> str:
    """
    Collapse ambiguous alleles with identical mature protein sequences.

    Groups the input alleles by their mature protein sequence (IMGT positions >= 1).
    For each group with more than one allele, the first allele in the group
    is kept as the representative and annotated with '[M]'.

    Alleles may be supplied as full names (e.g. 'DQA1*05:01') or as short
    field codes without the locus prefix (e.g. '05:01'); short codes are
    expanded automatically using the given locus.

    An allele is kept as its own separate entry (never collapsed) if:
      - its mature sequence is None (no reference sequence available), OR
      - its mature sequence contains '*' (one or more positions are unsequenced
        in the HLAtools alignment); unknown positions mean identity cannot be
        confirmed, so the allele is left unchanged.

    Args:
        alleles: List of allele names (ambiguous alternatives for one chromosome).
        locus:   HLA locus name.

    Returns:
        A '/'-joined GL string of representatives. Alleles with unique
        mature sequences are preserved as-is; groups with identical
        mature sequences are collapsed to 'FirstAllele[M]'.
    """
    def _normalize(allele):
        return allele if "*" in allele else f"{locus}*{allele}"

    normalized = [_normalize(a) for a in alleles]

    # Map allele -> mature sequence (prefix fallback via bridge)
    seq_map = {}
    for allele in normalized:
        seq = bridge.get_mature_sequence(allele)
        seq_map[allele] = seq

    # Group alleles by mature sequence, preserving input order
    seen_seqs = {}   # mature_seq -> first allele with that sequence
    groups = {}      # first_allele -> [all alleles sharing that sequence]
    for allele in normalized:
        seq = seq_map[allele]
        if seq is None or "*" in seq:
            # None  → no reference sequence available
            # '*'   → unsequenced positions present; cannot confirm identity
            # Either way: keep as its own separate entry, never collapse
            groups[allele] = [allele]
            continue
        if seq not in seen_seqs:
            seen_seqs[seq] = allele
            groups[allele] = [allele]
        else:
            groups[seen_seqs[seq]].append(allele)

    # Build output: representative for each unique mature sequence
    representatives = []
    for rep, members in groups.items():
        if len(members) > 1:
            representatives.append(f"{rep}[M]")
        else:
            representatives.append(rep)

    return "/".join(representatives)


def redux_gl_string(genotype: str, locus: str, redux_type: str = "M") -> str:
    """
    Apply a redux to each ambiguous haplotype group in a GL genotype string.

    Currently supports redux_type='M' (mature protein redux).

    Args:
        genotype:   Full GL genotype string, e.g. 'DQA1*05:01/05:05+DQA1*01:01'.
        locus:      HLA locus, used for sequence lookups.
        redux_type: Type of reduction to apply (currently only 'M').

    Returns:
        Reduxed GL string with the same '+' structure but each '/'
        group collapsed where appropriate.
    """
    if redux_type != "M":
        raise NotImplementedError(f"Redux type '{redux_type}' is not yet implemented")

    haplotypes = parse_gl_genotype(genotype, locus)
    reduxed_haps = []
    for hap in haplotypes:
        reduxed_haps.append(mature_protein_redux(hap, locus))
    return "+".join(reduxed_haps)


In [12]:
print("Test 7: M Redux — DQA1*05:xx group (differ only in leader / exon 1)")
print("=" * 65)

locus = "DQA1"
ambiguous_group = [
    "05:01",
    "05:05",
    "05:09",
    "05:11",
    "05:13",
    "05:35",
    "05:38",
    "05:41",
]

# Expand short codes to full allele names for bridge lookups
expanded_group = [f"{locus}*{a}" if "*" not in a else a for a in ambiguous_group]

print(f"\nLocus: {locus}")
print(f"Input alleles ({len(ambiguous_group)}):")
print("  " + " / ".join(ambiguous_group))

# Show full vs mature sequence lengths to confirm leader differs
full_alignment = bridge.get_alignment(locus)
mature_alignment = bridge.get_mature_alignment(locus)

print("\nFull vs mature sequence comparison (first 3 alleles):")
for allele in expanded_group[:3]:
    full_seq = bridge.get_sequence(allele)
    mat_seq = bridge.get_mature_sequence(allele)
    if full_seq and mat_seq:
        print(f"  {allele}:  full={len(full_seq)} aa,  mature={len(mat_seq)} aa  "
              f"(leader={len(full_seq)-len(mat_seq)} aa)")

# Show which alleles share the same mature sequence
print("\nMature sequences per allele (first 20 aa shown):")
mature_seqs = {}
for allele in expanded_group:
    seq = bridge.get_mature_sequence(allele)
    mature_seqs[allele] = seq
    preview = seq[:20] if seq else "NOT FOUND"
    print(f"  {allele}: {preview}...")

unique_mature = set(s for s in mature_seqs.values() if s is not None)
print(f"\nDistinct mature sequences in this group: {len(unique_mature)}")

# Apply the redux — pass short codes; mature_protein_redux expands them via locus
reduxed = mature_protein_redux(ambiguous_group, locus)
print(f"\nM redux result: {reduxed}")

# ── Sub-test: two alleles only — completely collapsed ────────────────────────
print("\n--- Sub-test: only DQA1*05:01 and DQA1*05:05 ---")
pair_result = mature_protein_redux(["05:01", "05:05"], "DQA1")
print(f"  Input:  05:01 / 05:05")
print(f"  Result: {pair_result}")
assert pair_result == "DQA1*05:01[M]", f"Expected DQA1*05:01[M], got {pair_result}"
print("  ✓ Collapsed to a single representative")

# ── Sub-test: missing reference sequence stays separate ──────────────────────
# An allele that has no sequence in HLAtools cannot be evaluated for mature-protein
# equivalence. It must remain in the output unchanged rather than being silently
# dropped. DQA1*99:99 is used as a stand-in for any allele with no reference data.
print("\n--- Sub-test: allele with no reference sequence left separate ---")
missing_result = mature_protein_redux(["05:01", "05:05", "99:99"], "DQA1")
print(f"  Input:  05:01 / 05:05 / 99:99  (99:99 has no reference sequence)")
print(f"  Result: {missing_result}")
assert missing_result == "DQA1*05:01[M]/DQA1*99:99", \
    f"Expected DQA1*05:01[M]/DQA1*99:99, got {missing_result}"
print("  ✓ 99:99 preserved as separate allele")


Test 7: M Redux — DQA1*05:xx group (differ only in leader / exon 1)

Locus: DQA1
Input alleles (8):
  05:01 / 05:05 / 05:09 / 05:11 / 05:13 / 05:35 / 05:38 / 05:41



Full vs mature sequence comparison (first 3 alleles):
  DQA1*05:01:  full=254 aa,  mature=231 aa  (leader=23 aa)
  DQA1*05:05:  full=254 aa,  mature=231 aa  (leader=23 aa)
  DQA1*05:09:  full=254 aa,  mature=231 aa  (leader=23 aa)

Mature sequences per allele (first 20 aa shown):
  DQA1*05:01: EDIVADHVASYGVNLYQSYG...
  DQA1*05:05: EDIVADHVASYGVNLYQSYG...
  DQA1*05:09: KDIVADHVASYGVNLYQSYG...
  DQA1*05:11: EDIVADHVASYGVNLYQSYG...
  DQA1*05:13: EDIVADHVASYGVNLYQSYG...
  DQA1*05:35: *****DHVASYGVNLYQSYG...
  DQA1*05:38: EDIVADHVASYGVNLYQSYG...
  DQA1*05:41: *****DHVASYGVNLYQSYG...

Distinct mature sequences in this group: 6

M redux result: DQA1*05:01[M]/DQA1*05:09/DQA1*05:11/DQA1*05:35/DQA1*05:38/DQA1*05:41

--- Sub-test: only DQA1*05:01 and DQA1*05:05 ---
  Input:  05:01 / 05:05
  Result: DQA1*05:01[M]
  ✓ Collapsed to a single representative

--- Sub-test: allele with no reference sequence left separate ---
  Input:  05:01 / 05:05 / 99:99  (99:99 has no reference sequence)
  Result: D

## Test 8: Identical AAMM (Amino Acid Mismatch Match)

Given ambiguous donor and recipient genotypes, there may be several possible true phasings. The **identical AAMM** check answers: *does it matter which phasing is real?* If every combination of (donor haplotype 1, donor haplotype 2) × (recipient haplotype 1, recipient haplotype 2) yields the same set of amino acid mismatches, then clinical assessment is unambiguous regardless of the actual allele phasing.

> **BDD coverage**: transplant mismatch and identical AAMM scenarios (unambiguous, ambiguous recipient, ambiguous donor) are run as a formal Gherkin suite in [Test 9](#test-9-bdd-feature-tests) below.

In [13]:
import itertools


def compute_aamm(donor_alleles: list, recipient_alleles: list) -> frozenset:
    """
    Compute the Amino Acid Mismatch (AAMM) set for one specific phasing.

    An AA position is a mismatch if the donor amino acid at that position
    differs from every recipient amino acid at that same position.

    Args:
        donor_alleles:     List of two allele names representing the donor diplotype
                           (one allele per chromosome, already resolved to a single
                           expressed allele each — no '/' ambiguity).
        recipient_alleles: List of two allele names for the recipient.

    Returns:
        frozenset of (seq_position, donor_aa) tuples for each mismatched position.
        seq_position is 1-based.
    """
    donor_seqs = [bridge.get_sequence(a) for a in donor_alleles]
    recip_seqs  = [bridge.get_sequence(a) for a in recipient_alleles]

    if any(s is None for s in donor_seqs + recip_seqs):
        return None  # Sequences unavailable — cannot assess

    seq_len = min(len(s) for s in donor_seqs + recip_seqs)
    aamm = set()
    for pos in range(seq_len):
        recip_aas = {s[pos] for s in recip_seqs}
        for d_seq in donor_seqs:
            donor_aa = d_seq[pos]
            if donor_aa not in recip_aas:
                aamm.add((pos + 1, donor_aa))

    return frozenset(aamm)


def check_identical_aamm(donor_genotype: str, recipient_genotype: str, locus: str) -> dict:
    """
    Determine whether all possible phasing combinations yield the same AAMM set.

    For each donor haplotype group (ambiguous '/' alternatives) and recipient
    haplotype group, one expressed allele is picked per chromosome.
    All combinations are enumerated via itertools.product.

    Args:
        donor_genotype:     GL string for donor, e.g. 'DPB1*04:01/DPB1*04:02+DPB1*02:01'.
        recipient_genotype: GL string for recipient.
        locus:              HLA locus.

    Returns:
        Dict with keys:
          'identical_aamm' (bool) — True if all combinations share the same AAMM set
          'aamm_sets'             — list of (donor_combo, recip_combo, aamm_set) per combination
          'unique_aamm_count'     — number of distinct AAMM sets seen
          'common_aamm'           — the AAMM set if identical, else None
    """
    donor_haps = parse_gl_genotype(donor_genotype, locus)
    recip_haps  = parse_gl_genotype(recipient_genotype, locus)

    # donor_haps is a list of chromosome groups; each group is a list of ambiguous alternatives
    # We need exactly 2 chromosomes (diploid)
    if len(donor_haps) != 2 or len(recip_haps) != 2:
        raise ValueError("Both donor and recipient genotypes must be diploid (exactly one '+').")

    # Filter null alleles from each group
    def expressed(group):
        return [a for a in group if not is_null_allele(a)] or group

    d_chr1_opts = expressed(donor_haps[0])
    d_chr2_opts = expressed(donor_haps[1])
    r_chr1_opts = expressed(recip_haps[0])
    r_chr2_opts = expressed(recip_haps[1])

    aamm_sets = []
    for d1, d2, r1, r2 in itertools.product(d_chr1_opts, d_chr2_opts, r_chr1_opts, r_chr2_opts):
        aamm = compute_aamm([d1, d2], [r1, r2])
        aamm_sets.append({
            "donor_combo":     (d1, d2),
            "recipient_combo": (r1, r2),
            "aamm":            aamm,
        })

    unique_aamm = {entry["aamm"] for entry in aamm_sets if entry["aamm"] is not None}
    identical = len(unique_aamm) == 1

    return {
        "donor_genotype":     donor_genotype,
        "recipient_genotype": recipient_genotype,
        "locus":              locus,
        "identical_aamm":     identical,
        "unique_aamm_count":  len(unique_aamm),
        "common_aamm":        next(iter(unique_aamm)) if identical else None,
        "aamm_sets":          aamm_sets,
    }


print("AAMM functions loaded")


AAMM functions loaded


In [14]:
print("Test 8: Identical AAMM Check")
print("=" * 55)

# ── Test 8a: unambiguous genotypes ──────────────────────────────────────────
print("\nTest 8a: Unambiguous genotypes (no ambiguity, single combination)")
print("  Donor:     DPB1*126:01+DPB1*105:01")
print("  Recipient: DPB1*04:01+DPB1*04:02")
result_a = check_identical_aamm(
    "DPB1*126:01+DPB1*105:01",
    "DPB1*04:01+DPB1*04:02",
    "DPB1",
)
print(f"  Identical AAMM: {result_a['identical_aamm']}  "
      f"(unique sets: {result_a['unique_aamm_count']})")
if result_a["common_aamm"] is not None:
    sorted_aamm = sorted(result_a["common_aamm"])
    print(f"  AAMM positions ({len(sorted_aamm)} total): {[pos for pos, _ in sorted_aamm]}")
else:
    for entry in result_a["aamm_sets"]:
        d1, d2 = entry["donor_combo"]
        r1, r2 = entry["recipient_combo"]
        n = len(entry["aamm"]) if entry["aamm"] is not None else "?"
        print(f"    D:({d1},{d2}) vs R:({r1},{r2}) -> {n} mismatched positions")

# ── Test 8b: ambiguous recipient (04:02 and 03:01 are NOT protein-identical) ─────
# DPB1*04:02 and DPB1*03:01 have different protein sequences (they differ at
# position 207). The two recipient phasings therefore produce different AAMM sets,
# so identical_aamm is correctly False.
print("\nTest 8b: Ambiguous recipient where the two alternatives are NOT protein-identical")
print("  Donor:     DPB1*126:01+DPB1*105:01")
print("  Recipient: DPB1*04:01+DPB1*04:02/03:01")
print("  (DPB1*04:02 and DPB1*03:01 differ at position 207 — not protein-identical)")
result_b = check_identical_aamm(
    "DPB1*126:01+DPB1*105:01",
    "DPB1*04:01+DPB1*04:02/03:01",
    "DPB1",
)
print(f"  Identical AAMM: {result_b['identical_aamm']}  "
      f"(unique sets: {result_b['unique_aamm_count']})")
print(f"  Combinations evaluated: {len(result_b['aamm_sets'])}")
if result_b["common_aamm"] is not None:
    sorted_aamm = sorted(result_b["common_aamm"])
    print(f"  AAMM positions ({len(sorted_aamm)} total): {[pos for pos, _ in sorted_aamm]}")
else:
    for entry in result_b["aamm_sets"]:
        d1, d2 = entry["donor_combo"]
        r1, r2 = entry["recipient_combo"]
        n = len(entry["aamm"]) if entry["aamm"] is not None else "?"
        sorted_positions = sorted(entry["aamm"]) if entry["aamm"] else []
        print(f"    D:({d1},{d2}) vs R:({r1},{r2}) -> {n} mismatched positions: "
              f"{[pos for pos, _ in sorted_positions]}")

# ── Test 8c: ambiguous donor — identical AAMM regardless of phasing ──────────
# Donor chr1 is ambiguous (04:01 or 126:01); chr2 is ambiguous (04:02 or 105:01).
# DPB1*04:01 and DPB1*126:01 are protein-identical; DPB1*04:02 and DPB1*105:01
# are protein-identical. Therefore every phasing combination yields the same
# (empty) AAMM set — identical_aamm is True.
print("\nTest 8c: Ambiguous donor — identical AAMM regardless of phasing")
print("  Donor:     DPB1*04:01/126:01+DPB1*04:02/105:01")
print("  Recipient: DPB1*126:01+DPB1*105:01")
result_c = check_identical_aamm(
    "DPB1*04:01/126:01+DPB1*04:02/105:01",
    "DPB1*126:01+DPB1*105:01",
    "DPB1",
)
print(f"  Identical AAMM: {result_c['identical_aamm']}  "
      f"(unique sets: {result_c['unique_aamm_count']})")
print(f"  Combinations evaluated: {len(result_c['aamm_sets'])}")
if result_c["common_aamm"] is not None:
    sorted_aamm = sorted(result_c["common_aamm"])
    print(f"  Common AAMM ({len(sorted_aamm)} positions): {[pos for pos, _ in sorted_aamm]}")
else:
    for entry in result_c["aamm_sets"]:
        d1, d2 = entry["donor_combo"]
        r1, r2 = entry["recipient_combo"]
        n = len(entry["aamm"]) if entry["aamm"] is not None else "?"
        print(f"    D:({d1},{d2}) vs R:({r1},{r2}) -> {n} mismatched positions")


Test 8: Identical AAMM Check

Test 8a: Unambiguous genotypes (no ambiguity, single combination)
  Donor:     DPB1*126:01+DPB1*105:01
  Recipient: DPB1*04:01+DPB1*04:02
  Identical AAMM: True  (unique sets: 1)
  AAMM positions (0 total): []

Test 8b: Ambiguous recipient where the two alternatives are NOT protein-identical
  Donor:     DPB1*126:01+DPB1*105:01
  Recipient: DPB1*04:01+DPB1*04:02/03:01
  (DPB1*04:02 and DPB1*03:01 differ at position 207 — not protein-identical)
  Identical AAMM: False  (unique sets: 2)
  Combinations evaluated: 2
    D:(DPB1*126:01,DPB1*105:01) vs R:(DPB1*04:01,DPB1*04:02) -> 0 mismatched positions: []
    D:(DPB1*126:01,DPB1*105:01) vs R:(DPB1*04:01,DPB1*03:01) -> 1 mismatched positions: [207]

Test 8c: Ambiguous donor — identical AAMM regardless of phasing
  Donor:     DPB1*04:01/126:01+DPB1*04:02/105:01
  Recipient: DPB1*126:01+DPB1*105:01
  Identical AAMM: True  (unique sets: 1)
  Combinations evaluated: 4
  Common AAMM (0 positions): []


## Test 9: BDD Feature Tests

The scenarios below are the exact cases from the repository's `tests/features/` BDD suite, run inline against the bridge already initialised in this session. Each line mirrors a Gherkin `Scenario` or `Scenario Outline` row.

In [15]:
class Feature:
    """Minimal Gherkin-style scenario runner for Jupyter notebooks."""

    _registry = []  # all instances, for cross-feature summary

    def __init__(self, name):
        self.name = name
        self.passed = []
        self.failed = []
        Feature._registry.append(self)
        print(f"Feature: {name}")

    def scenario(self, description, fn):
        try:
            fn()
            self.passed.append(description)
            print(f"  ✓  {description}")
        except Exception as exc:
            self.failed.append((description, exc))
            print(f"  ✗  {description}")
            print(f"       {type(exc).__name__}: {exc}")


def _eq(actual, expected):
    if actual != expected:
        raise AssertionError(f"got {actual!r}, expected {expected!r}")


def _none(value):
    if value is not None:
        raise AssertionError(f"expected None, got {value!r}")


def bdd_summary():
    """Print a combined summary across all Feature instances."""
    n_pass = sum(len(f.passed) for f in Feature._registry)
    n_fail = sum(len(f.failed) for f in Feature._registry)
    total = n_pass + n_fail
    print(f"\n{'=' * 60}")
    print(f"BDD summary  —  {total} scenarios: {n_pass} passed, {n_fail} failed")
    if n_fail:
        for f in Feature._registry:
            for name, exc in f.failed:
                print(f"  FAIL [{f.name}] {name}")
                print(f"       {type(exc).__name__}: {exc}")
    print("=" * 60)


# Reset registry so the cell is idempotent on re-run
Feature._registry.clear()
print("BDD runner ready.")


BDD runner ready.


In [16]:
# ── Feature: HLAtools Alignment Bridge ───────────────────────────────────────
f1 = Feature("HLAtools Alignment Bridge")

f1.scenario("Bridge reports itself as available",
    lambda: _eq(bridge.is_available, True))

for allele, expected_len in [
    ("DPB1*04:01:01:01", 258),
    ("DPB1*04:02:01:01", 258),
    ("DQA1*05:01",       254),
]:
    f1.scenario(
        f"Full sequence: {allele} → {expected_len} aa",
        lambda a=allele, n=expected_len: _eq(len(bridge.get_sequence(a)), n),
    )

for allele, leader_len in [
    ("DQA1*05:01", 23),
]:
    f1.scenario(
        f"Mature sequence: {allele} is {leader_len} aa shorter than full (leader peptide)",
        lambda a=allele, l=leader_len: _eq(
            len(bridge.get_sequence(a)) - len(bridge.get_mature_sequence(a)), l
        ),
    )

for a1, a2, n_mm in [
    ("DPB1*04:01:01:01", "DPB1*04:01:01:01", 0),
    ("DPB1*04:01:01:01", "DPB1*04:02:01:01", 4),
    ("DPB1*04:01",       "DPB1*04:02",       4),
]:
    f1.scenario(
        f"Sequence comparison: {a1} vs {a2} → {n_mm} mismatch(es)",
        lambda x=a1, y=a2, expected=n_mm: _eq(
            sum(1 for c1, c2 in zip(bridge.get_sequence(x), bridge.get_sequence(y)) if c1 != c2),
            expected,
        ),
    )

def _check_position_mapping():
    pm = bridge.get_position_mapping("DPB1")
    _eq(sum(1 for p in pm.values() if p < 1),  29)
    _eq(sum(1 for p in pm.values() if p >= 1), 244)

f1.scenario("DPB1 position mapping: 29 leader positions and 244 mature positions",
    _check_position_mapping)

for invalid in ["DPB1", ""]:
    f1.scenario(
        f"Invalid allele {invalid!r} returns no sequence",
        lambda a=invalid: _none(bridge.get_sequence(a)),
    )

print()

# ── Feature: Mature Protein Redux (M Redux) ───────────────────────────────────
f2 = Feature("Mature Protein Redux (M Redux)")

# Groups are specified with short field codes (no locus prefix); the locus is
# passed separately. mature_protein_redux expands short codes automatically.
# Full 4-field allele names (containing '*') are passed through unchanged.
for group, locus, expected in [
    # Same mature protein → collapse to [M]  (DQA1*05:05 shares mature seq with *05:01)
    ("05:01/05:05",                                    "DQA1", "DQA1*05:01[M]"),
    # Single allele — no collapsing
    ("05:01",                                          "DQA1", "DQA1*05:01"),
    # Different mature proteins — kept separate
    ("05:01/05:09",                                    "DQA1", "DQA1*05:01/DQA1*05:09"),
    ("05:01/05:11",                                    "DQA1", "DQA1*05:01/DQA1*05:11"),
    ("05:01/05:38",                                    "DQA1", "DQA1*05:01/DQA1*05:38"),
    # Full 8-allele group — 05:01/05:05/05:13 collapse to [M]; others kept separate
    (
        "05:01/05:05/05:09/05:11/05:13/05:35/05:38/05:41",
        "DQA1",
        "DQA1*05:01[M]/DQA1*05:09/DQA1*05:11/DQA1*05:35/DQA1*05:38/DQA1*05:41",
    ),
    # No reference sequence (bridge returns None) → kept separate
    ("05:01/05:05/99:99",                              "DQA1", "DQA1*05:01[M]/DQA1*99:99"),
    # Unsequenced positions in mature sequence ('*') → kept separate even if known suffix matches
    # DQA1*05:35 has a reference but its mature seq starts with ***** (unsequenced positions)
    ("05:01/05:35",                                    "DQA1", "DQA1*05:01/DQA1*05:35"),
    # DQB1: allele with unsequenced positions stays separate from complete allele
    ("DQB1*05:01:01:01/DQB1*05:01:02",                "DQB1", "DQB1*05:01:01:01/DQB1*05:01:02"),
    (
        "DQB1*05:01:01:01/DQB1*05:01:02/DQB1*05:01:03",
        "DQB1",
        "DQB1*05:01:01:01/DQB1*05:01:02/DQB1*05:01:03",
    ),
    # DRB1: same pattern
    (
        "DRB1*01:01:01:01/DRB1*01:01:03",
        "DRB1",
        "DRB1*01:01:01:01/DRB1*01:01:03",
    ),
]:
    allele_list = [a.strip() for a in group.split("/")]
    f2.scenario(
        f"M redux [{locus}]: {group[:55] + '…' if len(group) > 55 else group}  →  {expected}",
        lambda al=allele_list, loc=locus, exp=expected: _eq(
            mature_protein_redux(al, loc), exp
        ),
    )


Feature: HLAtools Alignment Bridge
  ✓  Bridge reports itself as available
  ✓  Full sequence: DPB1*04:01:01:01 → 258 aa
  ✓  Full sequence: DPB1*04:02:01:01 → 258 aa
  ✓  Full sequence: DQA1*05:01 → 254 aa
  ✓  Mature sequence: DQA1*05:01 is 23 aa shorter than full (leader peptide)
  ✓  Sequence comparison: DPB1*04:01:01:01 vs DPB1*04:01:01:01 → 0 mismatch(es)
  ✓  Sequence comparison: DPB1*04:01:01:01 vs DPB1*04:02:01:01 → 4 mismatch(es)
  ✓  Sequence comparison: DPB1*04:01 vs DPB1*04:02 → 4 mismatch(es)
  ✓  DPB1 position mapping: 29 leader positions and 244 mature positions
  ✓  Invalid allele 'DPB1' returns no sequence
  ✓  Invalid allele '' returns no sequence

Feature: Mature Protein Redux (M Redux)
  ✓  M redux [DQA1]: 05:01/05:05  →  DQA1*05:01[M]
  ✓  M redux [DQA1]: 05:01  →  DQA1*05:01
  ✓  M redux [DQA1]: 05:01/05:09  →  DQA1*05:01/DQA1*05:09
  ✓  M redux [DQA1]: 05:01/05:11  →  DQA1*05:01/DQA1*05:11
  ✓  M redux [DQA1]: 05:01/05:38  →  DQA1*05:01/DQA1*05:38
  ✓  M redux [

  ✓  M redux [DQB1]: DQB1*05:01:01:01/DQB1*05:01:02  →  DQB1*05:01:01:01/DQB1*05:01:02
  ✓  M redux [DQB1]: DQB1*05:01:01:01/DQB1*05:01:02/DQB1*05:01:03  →  DQB1*05:01:01:01/DQB1*05:01:02/DQB1*05:01:03


  ✓  M redux [DRB1]: DRB1*01:01:01:01/DRB1*01:01:03  →  DRB1*01:01:01:01/DRB1*01:01:03


In [17]:
# ── Feature: Transplant Mismatch Analysis ────────────────────────────────────
f3 = Feature("Transplant Mismatch Analysis")

# Scenario Outline: mismatch counts
for donor, recipient, locus, expected_count in [
    ("DPB1*04:01+DPB1*03:01",            "DPB1*04:01+DPB1*03:01", "DPB1", 0),
    ("DPB1*04:01+DPB1*02:01",            "DPB1*04:01+DPB1*03:01", "DPB1", 1),
    ("DPB1*04:01+DPB1*02:01N",           "DPB1*04:01+DPB1*03:01", "DPB1", 0),
    ("DPB1*04:01/04:02+DPB1*02:01",      "DPB1*04:01+DPB1*03:01", "DPB1", 1),
]:
    f3.scenario(
        f"Mismatch count: donor={donor}  →  {expected_count} mismatch(es)",
        lambda d=donor, r=recipient, loc=locus, n=expected_count: _eq(
            analyze_transplant_mismatches(d, r, loc)["mismatch_count"], n
        ),
    )

# Scenario Outline: per-haplotype status
# Note: DPB1*04:02 resolves correctly to its own sequence (distinct from 04:01),
# so the ambiguous haplotype DPB1*04:01/DPB1*04:02 is "possible_match" — 04:01
# matches a recipient sequence but 04:02 does not (its full sequence differs from
# both 04:01 and 03:01 in the leader region).
for donor, recipient, locus, haplotype_str, expected_status in [
    ("DPB1*04:01+DPB1*02:01",  "DPB1*04:01+DPB1*03:01", "DPB1", "DPB1*04:01",                   "match"),
    ("DPB1*04:01+DPB1*02:01",  "DPB1*04:01+DPB1*03:01", "DPB1", "DPB1*02:01",                   "mismatch"),
    ("DPB1*04:01+DPB1*02:01N", "DPB1*04:01+DPB1*03:01", "DPB1", "DPB1*02:01N",                  "null"),
    ("DPB1*04:01/04:02+DPB1*02:01", "DPB1*04:01+DPB1*03:01", "DPB1", "DPB1*04:01/DPB1*04:02",  "possible_match"),
]:
    def _check_status(d=donor, r=recipient, loc=locus, hap=haplotype_str, exp=expected_status):
        result = analyze_transplant_mismatches(d, r, loc)
        for hap_result in result["haplotype_results"]:
            if "/".join(hap_result["alleles"]) == hap:
                _eq(hap_result["status"], exp)
                return
        raise AssertionError(f"Haplotype '{hap}' not found in results")

    f3.scenario(
        f"Haplotype status: [{haplotype_str}] → {expected_status}",
        _check_status,
    )

print()

# ── Feature: Identical AAMM ───────────────────────────────────────────────────
f4 = Feature("Identical AAMM (Amino Acid Mismatch Match)")

# Scenario Outline: identical flag and distinct set count
for donor, recipient, locus, identical, distinct in [
    ("DPB1*126:01+DPB1*105:01",           "DPB1*04:01+DPB1*04:02",         "DPB1", True,  1),
    ("DPB1*126:01+DPB1*105:01",           "DPB1*04:01+DPB1*04:02/03:01",   "DPB1", False, 2),
    # Test 8c: ambiguous donor — 04:01/126:01 are protein-identical; 04:02/105:01 are
    # protein-identical. All phasing combinations give the same AAMM set.
    ("DPB1*04:01/126:01+DPB1*04:02/105:01", "DPB1*126:01+DPB1*105:01",    "DPB1", True,  1),
    # Ambiguous chr1 with genuinely different proteins (01:01 vs 02:01) → two distinct
    # AAMM sets → False/2.
    ("DPB1*01:01:01:01/02:01:02:01+DPB1*03:01:01:01", "DPB1*04:01:01:01+DPB1*04:02:01:01", "DPB1", False, 2),
]:
    def _check_aamm(d=donor, r=recipient, loc=locus, exp_id=identical, exp_n=distinct):
        res = check_identical_aamm(d, r, loc)
        _eq(res["identical_aamm"],   exp_id)
        _eq(res["unique_aamm_count"], exp_n)

    f4.scenario(
        f"AAMM identical={identical}: donor={donor}  /  recip={recipient}",
        _check_aamm,
    )

# Concrete AAMM position count for the unambiguous case.
# DPB1*04:01 and DPB1*04:02 together cover all AA positions present in
# DPB1*126:01+DPB1*105:01 (the union of recipient AAs at every position
# includes the donor AA), so the AAMM set is empty → 0 positions.
def _check_aamm_positions():
    res = check_identical_aamm("DPB1*126:01+DPB1*105:01", "DPB1*04:01+DPB1*04:02", "DPB1")
    _eq(len(res["common_aamm"]), 0)

f4.scenario(
    "AAMM positions: DPB1*126:01+DPB1*105:01 vs DPB1*04:01+DPB1*04:02 → 0 positions",
    _check_aamm_positions,
)

print()
bdd_summary()


Feature: Transplant Mismatch Analysis
  ✓  Mismatch count: donor=DPB1*04:01+DPB1*03:01  →  0 mismatch(es)
  ✓  Mismatch count: donor=DPB1*04:01+DPB1*02:01  →  1 mismatch(es)
  ✓  Mismatch count: donor=DPB1*04:01+DPB1*02:01N  →  0 mismatch(es)
  ✓  Mismatch count: donor=DPB1*04:01/04:02+DPB1*02:01  →  1 mismatch(es)
  ✓  Haplotype status: [DPB1*04:01] → match
  ✓  Haplotype status: [DPB1*02:01] → mismatch
  ✓  Haplotype status: [DPB1*02:01N] → null
  ✓  Haplotype status: [DPB1*04:01/DPB1*04:02] → possible_match

Feature: Identical AAMM (Amino Acid Mismatch Match)
  ✓  AAMM identical=True: donor=DPB1*126:01+DPB1*105:01  /  recip=DPB1*04:01+DPB1*04:02
  ✓  AAMM identical=False: donor=DPB1*126:01+DPB1*105:01  /  recip=DPB1*04:01+DPB1*04:02/03:01
  ✓  AAMM identical=True: donor=DPB1*04:01/126:01+DPB1*04:02/105:01  /  recip=DPB1*126:01+DPB1*105:01
  ✓  AAMM identical=False: donor=DPB1*01:01:01:01/02:01:02:01+DPB1*03:01:01:01  /  recip=DPB1*04:01:01:01+DPB1*04:02:01:01
  ✓  AAMM positions: DP

## Summary

### What This Notebook Demonstrates
- **HLAtools integration**: Alignment data is loaded from the HLAtools R package via rpy2, providing authoritative protein sequences and IMGT position numbering for all major HLA loci
- **Accurate position mapping**: IMGT positions come directly from HLAtools data frame column headers — no interpolation needed
- **Correct leader/mature boundaries**: DPB1 shows 29 leader and 244 mature positions with a clean -1 to 1 boundary
- **Donor-recipient compatibility analysis**: Full genotype-level comparison across all allele combinations with per-position mismatch detail
- **CSV export**: Results include IMGT position numbers and leader/mature region annotations
- **Multi-locus support**: Tested across A, B, C, DRB1, and DPB1
- **Mature protein redux (M redux)**: Ambiguous alleles that differ only in the leader peptide (exon 1, IMGT positions < 1) are collapsed to a single representative annotated with `[M]`, since they present an identical surface protein
- **Identical AAMM check**: Given ambiguous donor and recipient genotypes, all possible phasing combinations are enumerated to determine whether the amino acid mismatch set is the same regardless of the true allele assignment
- **Flexible allele input**: GL strings accept a locus prefix on the first alternative only (e.g. `DPB1*04:01/04:02+DPB1*02:01`); standalone ambiguous groups accept short field codes without any locus prefix (e.g. `["05:01", "05:05"]`) alongside a single `locus` argument

### Technical Architecture
- `pyard.alignment_bridge.HLAToolsBridge` wraps the HLAtools R package via rpy2
- Alignment data is lazily loaded on first access and cached in-memory
- `parse_gl_genotype` normalises short allele codes within GL strings to full names before analysis
- `mature_protein_redux` normalises short field codes to full allele names using the supplied locus, then groups by mature protein sequence
- HLAtools is required; see the Dependency Check cell at the top for installation instructions